# Рекомендации для удержания клиентов

Здесь мы связываем прогноз ухода с персональными рекомендациями. Цель не просто назвать риск, а ответить на вопрос: **что именно предложить клиенту, чтобы повысить шанс его удержания**.

## Логика решения

Наша финальная бизнес-цепочка выглядит так:

- считаем риск ухода клиента;
- выделяем клиентов высокого риска;
- смотрим их любимые категории и историю покупок;
- исключаем проблемные категории с высоким уровнем возвратов;
- рекомендуем более надёжные товары и категории;
- формируем итоговую таблицу для удерживающей кампании.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 300)
sns.set_theme(style="whitegrid", context="talk")

BASE_DIR = Path("..")
DATA_PATH = BASE_DIR / "data.csv"
EVENTS_PATH = BASE_DIR / "events.csv"


In [ ]:
def parse_dt(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, utc=True, errors="coerce", format="mixed")


def safe_mode(series: pd.Series):
    mode = series.mode(dropna=True)
    if not mode.empty:
        return mode.iat[0]
    non_null = series.dropna()
    return non_null.iat[0] if len(non_null) else np.nan


## 1. Загружаем данные

In [ ]:
data_cols = [
    "order_id", "order_item_id", "user_id", "status", "gender", "created_at", "returned_at",
    "shipped_at", "delivered_at", "sale_price", "age", "state", "city", "traffic_source",
    "category", "department", "brand", "product_id", "product_name_clean", "warehouse_name", "is_loyal"
]
event_cols = [
    "id", "user_id", "session_id", "sequence_number", "created_at", "traffic_source",
    "browser", "uri", "event_type"
]

data = pd.read_csv(DATA_PATH, usecols=data_cols)
events = pd.read_csv(EVENTS_PATH, usecols=event_cols)

for col in ["created_at", "returned_at", "shipped_at", "delivered_at"]:
    data[col] = parse_dt(data[col])
events["created_at"] = parse_dt(events["created_at"])

data = data.drop_duplicates(subset=["order_item_id"]).copy()
events = events.drop_duplicates(subset=["id"]).copy()

data["city"] = data["city"].fillna("unknown")
data["brand"] = data["brand"].fillna("unknown")
data["is_returned"] = data["returned_at"].notna().astype(int)
data["created_at_naive"] = data["created_at"].dt.tz_localize(None)

events = events.dropna(subset=["user_id", "created_at"]).copy()
events["user_id"] = events["user_id"].astype(int)
events["created_at_naive"] = events["created_at"].dt.tz_localize(None)


## 2. Строим рабочую таблицу риска

Чтобы этот ноутбук был самодостаточным, здесь мы не подгружаем результат из предыдущего ноутбука, а считаем простой рабочий риск по тем же принципам:

- клиенты с повторной покупкой или признаком лояльности;
- окно истории `365` дней;
- для ранжирования используем понятные признаки поведения.

Для рекомендаций нам сейчас важнее не идеальная модель, а качественный список клиентов по степени риска.

In [ ]:
CUTOFF_DATE = pd.Timestamp("2025-06-01")
LOOKBACK_START = CUTOFF_DATE - pd.Timedelta(days=365)

orders = (
    data.groupby("order_id", as_index=False)
    .agg(
        user_id=("user_id", "first"),
        created_at=("created_at_naive", "min"),
        order_value=("sale_price", "sum"),
        items_count=("order_item_id", "nunique"),
        is_loyal=("is_loyal", "max"),
    )
    .dropna(subset=["user_id", "created_at"])
)

history_orders = orders[orders["created_at"] < CUTOFF_DATE].copy()
window_orders = history_orders[(history_orders["created_at"] >= LOOKBACK_START) & (history_orders["created_at"] < CUTOFF_DATE)].copy()
window_items = data[(data["created_at_naive"] >= LOOKBACK_START) & (data["created_at_naive"] < CUTOFF_DATE)].copy()
window_events = events[(events["created_at_naive"] >= LOOKBACK_START) & (events["created_at_naive"] < CUTOFF_DATE)].copy()

base_users = (
    history_orders.groupby("user_id", as_index=False)
    .agg(
        total_orders_before_cutoff=("order_value", "count"),
        loyal_flag=("is_loyal", "max"),
        first_order_at=("created_at", "min"),
        last_order_at=("created_at", "max"),
    )
)
base_users = base_users[(base_users["total_orders_before_cutoff"] >= 2) | (base_users["loyal_flag"] == True)].copy()
base_users["days_since_last_order"] = (CUTOFF_DATE - base_users["last_order_at"]).dt.days
base_users["customer_age_days"] = (CUTOFF_DATE - base_users["first_order_at"]).dt.days
base_users.head()


In [ ]:
order_features = (
    window_orders.groupby("user_id", as_index=False)
    .agg(
        orders_last_365d=("order_value", "count"),
        revenue_last_365d=("order_value", "sum"),
        avg_order_value_last_365d=("order_value", "mean"),
        avg_items_per_order_last_365d=("items_count", "mean"),
    )
)

item_features = (
    window_items.groupby("user_id", as_index=False)
    .agg(
        return_rate_items=("is_returned", "mean"),
        returned_items_count=("is_returned", "sum"),
        distinct_categories_last_365d=("category", "nunique"),
        distinct_brands_last_365d=("brand", "nunique"),
        main_department=("department", lambda s: safe_mode(s)),
        main_category=("category", lambda s: safe_mode(s)),
        second_category=("category", lambda s: s.value_counts().index[1] if s.nunique() > 1 else np.nan),
        main_brand=("brand", lambda s: safe_mode(s)),
        avg_price_last_365d=("sale_price", "mean"),
        gender=("gender", "first"),
        age=("age", "median"),
        state=("state", lambda s: safe_mode(s)),
        city=("city", lambda s: safe_mode(s)),
    )
)

window_events["is_product"] = window_events["event_type"].eq("product").astype(int)
window_events["is_cart"] = window_events["event_type"].eq("cart").astype(int)
window_events["is_purchase"] = window_events["event_type"].eq("purchase").astype(int)

event_features = (
    window_events.groupby("user_id", as_index=False)
    .agg(
        sessions_last_365d=("session_id", "nunique"),
        events_last_365d=("id", "count"),
        product_views_last_365d=("is_product", "sum"),
        cart_events_last_365d=("is_cart", "sum"),
        purchase_events_last_365d=("is_purchase", "sum"),
        last_event_at=("created_at_naive", "max"),
    )
)
event_features["days_since_last_event"] = (CUTOFF_DATE - event_features["last_event_at"]).dt.days
event_features["events_per_session_last_365d"] = event_features["events_last_365d"] / event_features["sessions_last_365d"].replace(0, np.nan)

customer_mart = (
    base_users
    .merge(order_features, on="user_id", how="left")
    .merge(item_features, on="user_id", how="left")
    .merge(event_features, on="user_id", how="left")
)
customer_mart.shape


In [ ]:
risk_features = customer_mart.copy()

for col in [
    "days_since_last_order", "days_since_last_event", "return_rate_items", "orders_last_365d",
    "sessions_last_365d", "events_last_365d", "revenue_last_365d", "purchase_events_last_365d"
]:
    risk_features[col] = risk_features[col].fillna(risk_features[col].median())

risk_features["risk_score"] = (
    0.30 * (risk_features["days_since_last_order"] / risk_features["days_since_last_order"].max()) +
    0.20 * (risk_features["days_since_last_event"] / risk_features["days_since_last_event"].max()) +
    0.15 * risk_features["return_rate_items"].clip(0, 1) +
    0.15 * (1 - risk_features["orders_last_365d"] / risk_features["orders_last_365d"].max()) +
    0.10 * (1 - risk_features["sessions_last_365d"] / risk_features["sessions_last_365d"].max()) +
    0.10 * (1 - risk_features["purchase_events_last_365d"] / risk_features["purchase_events_last_365d"].replace(0, np.nan).max())
)
risk_features["risk_score"] = risk_features["risk_score"].fillna(risk_features["risk_score"].median())
risk_features["risk_percentile"] = risk_features["risk_score"].rank(pct=True)

risk_features["risk_segment"] = pd.cut(
    risk_features["risk_percentile"],
    bins=[0, 0.5, 0.8, 1.0],
    labels=["Низкий риск", "Средний риск", "Высокий риск"],
    include_lowest=True,
)

risk_features[["user_id", "risk_score", "risk_percentile", "risk_segment"]].head()


## 3. Готовим каталог безопасных рекомендаций

Мы не хотим рекомендовать всё подряд. Поэтому сначала делим категории и товары на более безопасные и более рискованные.

In [ ]:
category_stats = (
    data.groupby(["department", "category"], as_index=False)
    .agg(
        order_items=("order_item_id", "count"),
        customers=("user_id", "nunique"),
        revenue=("sale_price", "sum"),
        avg_price=("sale_price", "mean"),
        return_rate=("is_returned", "mean"),
    )
)
category_stats = category_stats.query("order_items >= 500").copy()
category_stats["revenue_per_item"] = category_stats["revenue"] / category_stats["order_items"]
category_stats["safety_score"] = (1 - category_stats["return_rate"]) * category_stats["revenue_per_item"]

safe_categories = category_stats.sort_values(["return_rate", "safety_score"], ascending=[True, False]).copy()
risky_categories = category_stats.sort_values(["return_rate", "safety_score"], ascending=[False, False]).copy()

display(safe_categories.head(10))
display(risky_categories.head(10))


In [ ]:
product_catalog = (
    data.groupby(["product_id", "product_name_clean", "department", "category", "brand"], as_index=False)
    .agg(
        order_items=("order_item_id", "count"),
        customers=("user_id", "nunique"),
        revenue=("sale_price", "sum"),
        avg_price=("sale_price", "mean"),
        return_rate=("is_returned", "mean"),
    )
)
product_catalog = product_catalog.query("order_items >= 20").copy()
product_catalog["product_score"] = (1 - product_catalog["return_rate"]) * product_catalog["revenue"]
product_catalog = product_catalog.sort_values("product_score", ascending=False)
product_catalog.head(10)


## 4. Логика удерживающих рекомендаций

Будем действовать так:

- если любимая категория клиента не выглядит проблемной, рекомендуем безопасные товары внутри неё;
- если любимая категория проблемная, рекомендуем соседнюю или вторую по интересу категорию;
- если предпочтения клиента размыты, рекомендуем безопасную и прибыльную категорию его департамента.

Плюс каждому клиенту дадим короткое объяснение, почему именно такой совет выдан.

In [ ]:
safe_threshold = category_stats["return_rate"].quantile(0.35)
risky_threshold = category_stats["return_rate"].quantile(0.80)

category_return_map = category_stats.set_index("category")["return_rate"].to_dict()
safe_by_department = {
    dep: grp.sort_values(["return_rate", "safety_score"], ascending=[True, False])["category"].tolist()
    for dep, grp in category_stats.groupby("department")
}

top_products_by_category = {
    cat: grp.sort_values(["return_rate", "product_score"], ascending=[True, False]).head(5)
    for cat, grp in product_catalog.groupby("category")
}

def choose_target_category(row):
    main_category = row.get("main_category")
    second_category = row.get("second_category")
    main_department = row.get("main_department")
    main_return = category_return_map.get(main_category, np.nan)
    second_return = category_return_map.get(second_category, np.nan)

    if pd.notna(main_return) and main_return <= safe_threshold:
        return main_category, "Сохраняем любимую категорию клиента: по ней низкий риск возврата."

    if pd.notna(second_category) and pd.notna(second_return) and second_return <= safe_threshold:
        return second_category, "Основная категория выглядит рискованной, поэтому берём вторую по интересу и более безопасную."

    if main_department in safe_by_department and len(safe_by_department[main_department]) > 0:
        return safe_by_department[main_department][0], "Выбираем наиболее надёжную категорию внутри привычного департамента клиента."

    if pd.notna(main_category):
        return main_category, "Сохраняем основную категорию из-за нехватки лучших альтернатив."

    return np.nan, "Недостаточно истории для точного выбора категории."


def choose_product_for_category(category_name, banned_brand=None):
    if pd.isna(category_name) or category_name not in top_products_by_category:
        return None
    candidates = top_products_by_category[category_name].copy()
    if pd.notna(banned_brand):
        alt = candidates[candidates["brand"] != banned_brand]
        if len(alt):
            candidates = alt
    return candidates.iloc[0]


high_risk = risk_features[risk_features["risk_segment"] == "Высокий риск"].copy()
high_risk[["target_category", "recommendation_reason"]] = high_risk.apply(
    lambda row: pd.Series(choose_target_category(row)), axis=1
)

recommended_rows = []
for _, row in high_risk.iterrows():
    product = choose_product_for_category(row["target_category"], banned_brand=row.get("main_brand"))
    recommended_rows.append({
        "user_id": row["user_id"],
        "risk_score": row["risk_score"],
        "risk_percentile": row["risk_percentile"],
        "risk_segment": row["risk_segment"],
        "main_department": row.get("main_department"),
        "main_category": row.get("main_category"),
        "second_category": row.get("second_category"),
        "target_category": row["target_category"],
        "recommended_product_id": None if product is None else product["product_id"],
        "recommended_product_name": None if product is None else product["product_name_clean"],
        "recommended_brand": None if product is None else product["brand"],
        "recommended_price": None if product is None else product["avg_price"],
        "recommended_product_return_rate": None if product is None else product["return_rate"],
        "recommendation_reason": row["recommendation_reason"],
        "loyal_flag": row.get("loyal_flag"),
        "days_since_last_order": row.get("days_since_last_order"),
        "days_since_last_event": row.get("days_since_last_event"),
        "return_rate_items": row.get("return_rate_items"),
    })

retention_table = pd.DataFrame(recommended_rows)
retention_table.head()


In [ ]:
summary = pd.Series(
    {
        "eligible_customers": len(risk_features),
        "high_risk_customers": len(retention_table),
        "share_high_risk": round(len(retention_table) / len(risk_features), 4),
        "recommendations_ready": int(retention_table["recommended_product_id"].notna().sum()),
    }
).to_frame("value")
summary


In [ ]:
retention_table.sort_values("risk_score", ascending=False).head(15)


## 5. Как это показывать на защите

Этот блок нужно презентовать не как "мы что-то порекомендовали", а как понятную бизнес-логику:

1. Мы заранее находим клиентов с высоким риском ухода.
2. Для каждого клиента определяем, в какой категории лучше его возвращать.
3. Не рекомендуем проблемные категории с повышенным возвратом.
4. Даём конкретный товар и объяснение, почему он выбран.

Итоговая бизнес-формула звучит так:

**не просто предсказать уход, а дать следующему действию шанс удержать клиента.**

## 6. Во что это потом заворачивать

Финально это можно упаковать в три слоя:

- `Аналитический слой`: дашборды по продажам, клиентам, оттоку, логистике, возвратам.
- `ML-слой`: модель риска ухода и логика удерживающих рекомендаций.
- `Демонстрационный слой`: экран или таблица, где по клиенту видно риск, причину риска и рекомендованное действие.

На защите можно показать один понятный сценарий:

- клиент давно не покупал;
- у него вырос риск ухода;
- его любимая категория рискованная по возвратам;
- система предлагает более безопасную категорию и конкретный товар;
- бизнес получает готовый список для удерживающей кампании.

## Что дальше

После этого ноутбука у нас уже есть очень сильная логика проекта. Следующие разумные шаги:

- подготовить финальную витрину `клиент -> риск -> причина -> рекомендация`;
- продумать дашборд или демонстрационный экран;
- оформить итоговый рассказ для презентации и защиты.